# Tools for a Customer Outreach Campaign

In this lesson, you will learn more about Tools. You'll focus on three key elements of Tools:
- Versatility
- Fault Tolerance
- Caching

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
import os

warnings.filterwarnings('ignore')
os.environ["OTEL_SDK_DISABLED"] = "true"

- Import libraries, APIs and LLM
- [Serper](https://serper.dev)

In [3]:
from crewai import Agent, Task, Crew

18:36:09 - LiteLLM:WARNING: get_model_cost_map.py:271 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: _ssl.c:993: The handshake operation timed out. Falling back to local backup.


In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Creating Agents

In [7]:
sales_rep_agent = Agent(
    role="نماینده فروش",
    goal="شناسایی سرنخ‌های (lead) با ارزش بالا که با "
         "پروفایل مشتری ایده‌آل ما مطابقت دارند",
    backstory=(
        "به عنوان بخشی از تیم پویای فروش در CrewAI، "
        "مأموریت شما کاوش در فضای دیجیتال برای یافتن سرنخ‌های (lead) بالقوه است. "
        "مجهز به ابزارهای پیشرفته "
        "و ذهنیت استراتژیک، شما داده‌ها، "
        "روندها و تعاملات را تحلیل می‌کنید تا "
        "فرصت‌هایی را کشف کنید که دیگران ممکن است از دست بدهند. "
        "کار شما در هموار کردن مسیر "
        "برای تعاملات معنادار و پیشبرد رشد شرکت نقش حیاتی دارد."
    ),
    allow_delegation=False,
    verbose=True
)

In [8]:
lead_sales_rep_agent = Agent(
    role="نماینده ارشد فروش",
    goal="پرورش لیدها با ارتباطات شخصی‌سازی‌شده و قانع‌کننده",
    backstory=(
        "در اکوسیستم پرجنب‌وجوش بخش فروش CrewAI، "
        "شما به عنوان پل ارتباطی بین مشتریان بالقوه "
        "و راه‌حل‌های مورد نیازشان برجسته هستید. "
        "با ایجاد پیام‌های جذاب و شخصی‌سازی‌شده، "
        "نه تنها لیدها را از محصولات ما آگاه می‌کنید "
        "بلکه باعث می‌شوید احساس دیده شدن و شنیده شدن داشته باشند. "
        "نقش شما در تبدیل علاقه به عمل، "
        "و هدایت لیدها در مسیر از کنجکاوی تا تعهد، حیاتی است."
    ),
    allow_delegation=False,
    verbose=True
)

## Creating Tools

### crewAI Tools

In [10]:
from crewai_tools import DirectoryReadTool, \
                         FileReadTool, \
                         SerperDevTool

In [11]:
directory_read_tool = DirectoryReadTool(directory='./instructions')
file_read_tool = FileReadTool()
search_tool = SerperDevTool()

In [18]:
search_tool._run(query="علیرضا اخوان پور")

{'searchParameters': {'q': 'علیرضا اخوان پور',
  'type': 'search',
  'num': 10,
  'engine': 'google'},
 'organic': [{'title': 'علیرضا اخوان پور - کلاس\u200cویژن - class.vision',
   'link': 'https://class.vision/teacher/%D8%B9%D9%84%DB%8C%D8%B1%D8%B6%D8%A7-%D8%A7%D8%AE%D9%88%D8%A7%D9%86-%D9%BE%D9%88%D8%B1/',
   'snippet': 'علیرضا اخوان پور، متخصص برجسته در حوزه هوش مصنوعی و یادگیری عمیق، با بیش از 10 سال سابقه تدریس و فعالیت حرفه\u200cای، در حال حاضر به عنوان مدیر فنی مجموعه دانش ...',
   'position': 1},
  {'title': 'علیرضا اخوان\u200cپور',
   'link': 'https://maktabkhooneh.org/teacher/alireza-akhavan-1/',
   'snippet': 'علیرضا اخوان\u200cپور، متخصص برجسته در حوزه هوش مصنوعی و یادگیری عمیق، با بیش از ۱۰ سال سابقه تدریس و فعالیت حرفه\u200cای، یکی از چهره\u200cهای شناخته\u200cشده در این حوزه است.',
   'position': 2},
  {'title': 'علیرضا اخوان پور',
   'link': 'https://www.aparat.com/cplusplus',
   'snippet': 'مبانی برنامه سازی - جلسه 14 (حلقه با for و الگوریتمهای مبتنی بر حدس و بررسی) · ع

### Custom Tool
- Create a custom tool using crewAi's [BaseTool](https://docs.crewai.com/core-concepts/Tools/#subclassing-basetool) class

In [32]:
from crewai.tools import BaseTool

- Every Tool needs to have a `name` and a `description`.
- For simplicity and classroom purposes, `SentimentAnalysisTool` will return `positive` for every text.
- When running locally, you can customize the code with your logic in the `_run` function.

In [35]:
class SentimentAnalysisTool(BaseTool):
    name: str ="Sentiment Analysis Tool"
    description: str = ("Analyzes the sentiment of text "
         "to ensure positive and engaging communication.")
    
    def _run(self, text: str) -> str:
        # Your custom code tool goes here
        return "positive"

In [37]:
sentiment_analysis_tool = SentimentAnalysisTool()

In [39]:
from crewai.tools import BaseTool
import requests
# for create an account and credit: https://chat.avalai.ir/?ref=1LMQHOW
class AvalAISearchTool(BaseTool):
    name: str = "AvalAI Search"
    description: str = "Search using AvalAI Serper API"

    def _run(self, query: str):
        api_key = os.getenv("AVALAI_API_KEY")
        response = requests.post(
            "https://api.avalai.ir/v1/search/serper-search",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json"
            },
            json={
                "query": query,
                "max_results": 5
            }
        )

        return response.json()

In [29]:
search_tool2 = AvalAISearchTool()
search_tool2._run(query="علیرضا اخوان پور")

{'results': [{'title': 'علیرضا اخوان پور - کلاس\u200cویژن - class.vision',
   'url': 'https://class.vision/teacher/%D8%B9%D9%84%DB%8C%D8%B1%D8%B6%D8%A7-%D8%A7%D8%AE%D9%88%D8%A7%D9%86-%D9%BE%D9%88%D8%B1/',
   'snippet': 'علیرضا اخوان پور، متخصص برجسته در حوزه هوش مصنوعی و یادگیری عمیق، با بیش از 10 سال سابقه تدریس و فعالیت حرفه\u200cای، در حال حاضر به عنوان مدیر فنی مجموعه دانش ...',
   'date': None,
   'last_updated': None},
  {'title': 'علیرضا اخوان\u200cپور',
   'url': 'https://maktabkhooneh.org/teacher/alireza-akhavan-1/',
   'snippet': 'علیرضا اخوان\u200cپور، متخصص برجسته در حوزه هوش مصنوعی و یادگیری عمیق، با بیش از ۱۰ سال سابقه تدریس و فعالیت حرفه\u200cای، یکی از چهره\u200cهای شناخته\u200cشده در این حوزه است.',
   'date': None,
   'last_updated': None},
  {'title': 'علیرضا اخوان\u200cپور',
   'url': 'https://nikamooz.com/professors/alireza-akhavanpour/',
   'snippet': '۸ سال سابقه ی مدیر فنی در مجموعه دانش بنیان “شناسا” · توسعه دهنده پایتون و فعال در پروژه های بینایی ماشین با یادگ

## Creating Tasks

- The Lead Profiling Task is using crewAI Tools.

In [41]:
lead_profiling_task = Task(
    description=(
        "راهنماها و دستورالعمل‌های موجود برای صنعت {industry} را پیدا و مطالعه کنید. "
        "سپس یک تحلیل عمیق از {lead_name}، "
        "شرکتی در حوزه {industry} "
        "که اخیراً به راه‌حل‌های ما علاقه نشان داده، انجام دهید. "
        "از تمام منابع داده‌ای موجود استفاده کنید "
        "تا یک پروفایل جامع تهیه شود، "
        "با تمرکز بر تصمیم‌گیرندگان کلیدی، "
        "تحولات اخیر کسب‌وکار، و نیازهای بالقوه‌ای "
        "که با پیشنهادات ما همسو هستند. "
        "این وظیفه برای تنظیم مؤثر استراتژی تعامل ما بسیار حیاتی است.\n"
        "هیچ چیزی را حدس نزنید و "
        "تنها از اطلاعاتی استفاده کنید که کاملاً از صحت آن‌ها مطمئن هستید."
    ),
    expected_output=(
        "یک گزارش جامع درباره {lead_name}، "
        "شامل پیشینه شرکت، "
        "افراد کلیدی، دستاوردهای اخیر، و نیازهای شناسایی‌شده. "
        "همچنین دستورالعمل مرتبط با صنعت {industry} را که مطالعه شده "
        "به‌عنوان مرجع استراتژی تعامل ذکر کنید. "
        "حوزه‌هایی را که راه‌حل‌های ما می‌توانند ارزش‌آفرینی کنند برجسته کنید "
        "و استراتژی‌های تعامل شخصی‌سازی‌شده پیشنهاد دهید."
    ),
    tools=[directory_read_tool, file_read_tool, search_tool],
    agent=sales_rep_agent,
)

- The Personalized Outreach Task is using your custom Tool `SentimentAnalysisTool`, as well as crewAI's `SerperDevTool` (search_tool).

In [44]:
personalized_outreach_task = Task(
    description=(
        "با استفاده از اطلاعات و دستورالعمل‌های به‌دست‌آمده از "
        "گزارش پروفایل سرنخ (lead) برای {lead_name}، "
        "یک کمپین بازاریابی هدفمند "
        "با هدف {key_decision_maker}، "
        "{position} شرکت {lead_name} طراحی کنید. "
        "از تمپلت و راهنمای مرتبط با صنعت {industry} که در گزارش ذکر شده "
        "به‌عنوان چارچوب اصلی نگارش استفاده کنید. "
        "این کمپین باید به {milestone} اخیر آن‌ها بپردازد "
        "و نشان دهد راه‌حل‌های ما چگونه از اهدافشان حمایت می‌کند. "
        "ارتباط شما باید با فرهنگ و ارزش‌های شرکت {lead_name} همسو باشد "
        "و درک عمیقی از کسب‌وکار و نیازهای آن‌ها نشان دهد.\n"
        "هیچ چیزی را حدس نزنید و "
        "تنها از اطلاعاتی استفاده کنید که کاملاً از صحت آن‌ها مطمئن هستید."
    ),
    expected_output=(
        "مجموعه‌ای از پیش‌نویس‌های ایمیل شخصی‌سازی‌شده "
        "برای {lead_name}، "
        "با تمرکز ویژه بر {key_decision_maker}. "
        "هر پیش‌نویس باید بر اساس تمپلت صنعت {industry} نوشته شده باشد "
        "و شامل پیامی متقاعدکننده باشد که راه‌حل‌های ما را "
        "به دستاوردهای اخیر و اهداف آینده آن‌ها پیوند دهد. "
        "لحن نوشتار باید جذاب، حرفه‌ای "
        "و همسو با هویت سازمانی {lead_name} باشد."
    ),
    tools=[directory_read_tool, file_read_tool, sentiment_analysis_tool, search_tool],
    agent=lead_sales_rep_agent,
)

## Creating the Crew

In [78]:
crew = Crew(
    agents=[sales_rep_agent, 
            lead_sales_rep_agent],
    
    tasks=[lead_profiling_task, 
           personalized_outreach_task],
	
    verbose=True,
	memory=True
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [83]:
inputs = {
    "lead_name": "کلاس وِیژن",
    "industry": "پلتفرم آموزشی آنلاین",
    "key_decision_maker": "علیرضا اخوان پور",
    "position": "CEO",
    "milestone": "product launch"
}

result = crew.kickoff(inputs=inputs)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.2                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1ecd0a27-5e43-4dfa-85e6-7b9e7430b9ad                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: راهنماها و دستورالعمل‌های موجود برای صنعت پلتفرم آموزشی آنلاین را پیدا و مطالعه کنید. سپس یک تحلیل عمیق   │
│  از کلاس وِیژن، شرکتی در حوزه پلتفرم آموزشی آنلاین که اخیراً به راه‌حل‌های ما علاقه نشان داده، انجام دهید. از تمام  │
│  منابع داده‌ای موجود استفاده کنید تا یک پروفایل جامع تهیه شود، با تمرکز بر تصمیم‌گیرندگان کلیدی، تحولات اخیر      │
│  کسب‌وکار، و نیازهای بالقوه‌ای که با پیشنهادات ما همسو هستند. این وظیفه برای تنظیم مؤثر استراتژی تعامل ما بسیار   │
│  حیاتی است.                                                                                                     │
│  هیچ چیزی را حدس نزنید و تنها از اطلاعاتی استفاده کنید که کاملاً از صحت آن‌ها مطمئن هستید.                        │
│  ID: de5760d2-c590-427d-a8cd-d07d4e283e7e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: نماینده فروش                                                                                            │
│                                                                                                                 │
│  Task: راهنماها و دستورالعمل‌های موجود برای صنعت پلتفرم آموزشی آنلاین را پیدا و مطالعه کنید. سپس یک تحلیل عمیق   │
│  از کلاس وِیژن، شرکتی در حوزه پلتفرم آموزشی آنلاین که اخیراً به راه‌حل‌های ما علاقه نشان داده، انجام دهید. از تمام  │
│  منابع داده‌ای موجود استفاده کنید تا یک پروفایل جامع تهیه شود، با تمرکز بر تصمیم‌گیرندگان کلیدی، تحولات اخیر      │
│  کسب‌وکار، و نیازهای بالقوه‌ای که با پیشنهادات ما همسو هستند. این وظیفه برای تنظیم مؤثر استراتژی تعامل ما بسیار   │
│  حیاتی است.                                                                                                     │
│  هیچ چیزی را حدس نزنید و تنها از اطلاعاتی استفاده کنید که کاملاً از صحت آن‌ها مطمئن هستید.                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['کلاس ویژن شرکت پلتفرم آموزشی آنلاین', 'کلاس ویژن تصمیم\u200cگیرندگان کلیدی', 'کلاس ویژن    │
│  تحولات اخیر کسب\u200cوکار', 'نیازهای کلاس ویژن در آموزش آنلاین', 'راهنما و دستورالعمل صنعت پلتفرم ...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: نماینده ارشد فروش                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  موضوع: همکارى براى حمایت از رونمایی محصول جدید کلاس ویژن با راهکارهاى نوآورانه آموزش آنلاین                    │
│                                                                                                                 │
│  سلام آقای علیرضا اخوان پور عزیز،                                                                               │
│  امیدوارم حالتان خوب باشد.                                                                                      │
│                                                                                                                 │
│  ابتدا تبریک صمیمانه ما را بابت رونمایی محصول جدید شرکت کلاس ویژن پذیرا باشید. این گام مهم نشان دهنده تعهد شما  │
│  و تیم‌تان به نوآوری و بهبود کیفیت آموزش آنلاین است که ما در CrewAI به شدت هم‌راستا با این اهداف ارزشمند هستیم.   │
│                                                                                                                 │
│  با توجه به تمرکز کلاس ویژن بر ارائه آموزش‌های آنلاین شخصی‌سازی شده و تعاملی و توجه ویژه به فناوری‌های نوین،       │
│  معتقدیم راهکارهای ما می‌تواند نقشی کلیدی در ارتقای این محصول جدید ایفا کند.                                     │
│  راهکارهای حرفه‌ای CrewAI در این زمینه شامل:                                                                     │
│  - فناوری‌های هوش مصنوعی و یادگیری ماشینی برای شخصی‌سازی عمیق تجربه یادگیری و افزایش کیفیت آموزش،                 │
│  - ابزارهای تحلیل داده و داده‌کاوی برای بهینه‌سازی فرآیندهای آموزشی و بازاریابی و تصمیم‌گیری هوشمندانه،            │
│  - همکاری در توسعه محتوای تخصصی و تعاملی مطابق با نیازهای روز بازار،                                            │
│  - ارتقاء تجربه کاربری با استفاده از فناوری‌های تعاملی و تحلیل رفتار کاربران،                                    │
│                                                                                                                 │
│  ما مشتاقیم فرصتی برای ارائه دمو و نمونه‌های کاربردی این فناوری‌ها داشته باشیم و چگونگی هماهنگی آن‌ها با اهداف     │
│  کلاس ویژن را به صورت دقیق‌تر بررسی کنیم. باور داریم این همکاری می‌تواند ارزش افزوده قابل توجهی در تحقق اهداف     │
│  توسعه و موفقیت محصول جدید شما باشد.                                                                            │
│                                                                                                                 │
│  با احترام و آرزوی موفقیت‌های پیوسته،                                                                            │
│  [نام شما]                                                                                                      │
│  نماینده ارشد فروش                                                                                              │
│  CrewAI                                                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  موضوع: پیشنهاد همکاری برای افزایش کیفیت آموزش در کلاس ویژن و پشتیبانی از محصول جدید                            │
│                                                                                                                 │
│  سلام جناب آقای اخوان پور،          

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: با استفاده از اطلاعات و دستورالعمل‌های به‌دست‌آمده از گزارش پروفایل سرنخ (lead) برای کلاس وِیژن، یک کمپین    │
│  بازاریابی هدفمند با هدف علیرضا اخوان پور، CEO شرکت کلاس وِیژن طراحی کنید. از تمپلت و راهنمای مرتبط با صنعت      │
│  پلتفرم آموزشی آنلاین که در گزارش ذکر شده به‌عنوان چارچوب اصلی نگارش استفاده کنید. این کمپین باید به product     │
│  launch اخیر آن‌ها بپردازد و نشان دهد راه‌حل‌های ما چگونه از اهدافشان حمایت می‌کند. ارتباط شما باید با فرهنگ و      │
│  ارزش‌های شرکت کلاس وِیژن همسو باشد و درک عمیقی از کسب‌وکار و نیازهای آن‌ها نشان دهد.                               │
│  هیچ چیزی را حدس نزنید و تنها از اطلاعاتی استفاده کنید که کاملاً از صحت آن‌ها مطمئن هستید.                        │
│  Agent: نماینده ارشد فروش                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1ecd0a27-5e43-4dfa-85e6-7b9e7430b9ad                                                                       │
│  Final Output: موضوع: همکارى براى حمایت از رونمایی محصول جدید کلاس ویژن با راهکارهاى نوآورانه آموزش آنلاین      │
│                                                                                                                 │
│  سلام آقای علیرضا اخوان پور عزیز،                                                                               │
│  امیدوارم حالتان خوب باشد.                                                                                      │
│                                                                                                                 │
│  ابتدا تبریک صمیمانه ما را بابت رونمایی محصول جدید شرکت کلاس ویژن پذیرا باشید. این گام مهم نشان دهنده تعهد شما  │
│  و تیم‌تان به نوآوری و بهبود کیفیت آموزش آنلاین است که ما در CrewAI به شدت هم‌راستا با این اهداف ارزشمند هستیم.   │
│                                                                                                                 │
│  با توجه به تمرکز کلاس ویژن بر ارائه آموزش‌های آنلاین شخصی‌سازی شده و تعاملی و توجه ویژه به فناوری‌های نوین،       │
│  معتقدیم راهکارهای ما می‌تواند نقشی کلیدی در ارتقای این محصول جدید ایفا کند.                                     │
│  راهکارهای حرفه‌ای CrewAI در این زمینه شامل:                                                                     │
│  - فناوری‌های هوش مصنوعی و یادگیری ماشینی برای شخصی‌سازی عمیق تجربه یادگیری و افزایش کیفیت آموزش،                 │
│  - ابزارهای تحلیل داده و داده‌کاوی برای بهینه‌سازی فرآیندهای آموزشی و بازاریابی و تصمیم‌گیری هوشمندانه،            │
│  - همکاری در توسعه محتوای تخصصی و تعاملی مطابق با نیازهای روز بازار،                                            │
│  - ارتقاء تجربه کاربری با استفاده از فناوری‌های تعاملی و تحلیل رفتار کاربران،                                    │
│                                                                                                                 │
│  ما مشتاقیم فرصتی برای ارائه دمو و نمونه‌های کاربردی این فناوری‌ها داشته باشیم و چگونگی هماهنگی آن‌ها با اهداف     │
│  کلاس ویژن را به صورت دقیق‌تر بررسی کنیم. باور داریم این همکاری می‌تواند ارزش افزوده قابل توجهی در تحقق اهداف     │
│  توسعه و موفقیت محصول جدید شما باشد.                                                                            │
│                                                                                                                 │
│  با احترام و آرزوی موفقیت‌های پیوسته،                                                                            │
│  [نام شما]                                                                                                      │
│  نماینده ارشد فروش                                                                                              │
│  CrewAI                                                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  موضوع: پیشنهاد همکاری برای افزایش کیفیت آموزش در کلاس ویژن و پشتیبانی از محصول جدید                            │
│                                                                                                                 │
│  سلام جناب آقای اخوان پور،         

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:crewai.flow.flow:Error executing listener execute_plans: Connection error.


- Display the final result as Markdown.

In [85]:
from IPython.display import HTML
import markdown

html_content = markdown.markdown(result.raw)

HTML(f'''
<style>
    .rtl-content * {{
        direction: rtl !important;
        text-align: right !important;
    }}
</style>
<div class="rtl-content" style="
    direction: rtl;
    text-align: right; 
    font-family: Tahoma, Arial, sans-serif; 
    font-size: 15px; 
    line-height: 2;
    padding: 20px;
">
    {html_content}
</div>
''')


As you saw, the agent did not stop after the error and continued the workflow. However, the issue originates from the file reading tool, so in the next notebook (`07_03_tools_customer_outreach_v2.ipynb`) we will fix the tool to properly handle file encoding and avoid this error.
